#환경

In [ ]:
!pip -q install "qai-hub-models[facemap-3dmm]" mediapipe opencv-python-headless scikit-learn joblib
!wget -q -O /content/face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.9/70.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!mkdir -p /content/train_images
!ffmpeg -i /content/input.mp4 -vf fps=3 /content/train_images/frame_%05d.jpg


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

#다

##import +모델/헬퍼함수

In [ ]:
import os
import glob
import json
import joblib
import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

from qai_hub_models.models.facemap_3dmm.model import FaceMap_3DMM

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode


def create_face_landmarker(model_path="/content/face_landmarker.task", num_faces=1):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)


def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def infer_mediapipe_teacher(landmarker, image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = landmarker.detect(mp_image)

    if not result.face_landmarks or not result.face_blendshapes:
        return None

    h, w = image_rgb.shape[:2]
    lms = result.face_landmarks[0]
    pts = np.array([[lm.x * w, lm.y * h, lm.z * w] for lm in lms], dtype=np.float32)

    cats = result.face_blendshapes[0]
    score_map = {c.category_name: float(c.score) for c in cats}

    matrix = None
    if result.facial_transformation_matrixes:
        matrix = np.array(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(-1)

    return pts, score_map, matrix


def get_face_bbox_from_landmarks(pts, image_shape, padding=0.25, make_square=True):
    h, w = image_shape[:2]

    xs = pts[:, 0]
    ys = pts[:, 1]

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0
    bh = y1 - y0

    x0 -= bw * padding
    x1 += bw * padding
    y0 -= bh * padding
    y1 += bh * padding

    if make_square:
        cx = (x0 + x1) / 2.0
        cy = (y0 + y1) / 2.0
        size = max(x1 - x0, y1 - y0)
        x0 = cx - size / 2.0
        x1 = cx + size / 2.0
        y0 = cy - size / 2.0
        y1 = cy + size / 2.0

    x0 = int(np.clip(np.floor(x0), 0, w - 1))
    y0 = int(np.clip(np.floor(y0), 0, h - 1))
    x1 = int(np.clip(np.ceil(x1), 0, w - 1))
    y1 = int(np.clip(np.ceil(y1), 0, h - 1))

    return x0, y0, x1, y1


def infer_qualcomm_265(model, image_rgb, bbox):
    x0, y0, x1, y1 = bbox
    crop = image_rgb[y0:y1 + 1, x0:x1 + 1]

    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy(crop).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        raw265 = model(inp)[0].cpu().numpy().astype(np.float32)

    return raw265


def collect_image_paths(root):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(paths)


##데이터셋


In [ ]:
TRAIN_DIR = "/content/train_images"

image_paths = collect_image_paths(TRAIN_DIR)
print("num images:", len(image_paths))

q_model = FaceMap_3DMM.from_pretrained()

X_list = []
Y_list = []
M_list = []
blendshape_names = None
used_paths = []
skipped = []

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(image_paths):
        try:
            image_rgb = read_rgb(path)

            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                skipped.append((path, "mediapipe_failed"))
                continue

            pts, score_map, matrix = teacher

            if blendshape_names is None:
                blendshape_names = sorted(score_map.keys())

            y = np.array([score_map[name] for name in blendshape_names], dtype=np.float32)

            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                skipped.append((path, "qualcomm_failed"))
                continue

            X_list.append(x)
            Y_list.append(y)
            if matrix is not None:
                M_list.append(matrix)
            used_paths.append(path)

            if (i + 1) % 100 == 0:
                print(f"processed {i+1}/{len(image_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))

X = np.stack(X_list).astype(np.float32)
Y = np.stack(Y_list).astype(np.float32)
M = np.stack(M_list).astype(np.float32) if len(M_list) == len(X_list) else None

print("X shape:", X.shape)  # [N, 265]
print("Y shape:", Y.shape)  # [N, 52]
print("skipped:", len(skipped))

np.savez(
    "/content/q265_to_mp52_dataset.npz",
    X=X,
    Y=Y,
    M=M if M is not None else np.array([]),
    blendshape_names=np.array(blendshape_names, dtype=object),
    used_paths=np.array(used_paths, dtype=object),
)

with open("/content/skipped.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("saved: /content/q265_to_mp52_dataset.npz")


num images: 562


processed 100/562
processed 200/562
processed 300/562
processed 400/562
processed 500/562
X shape: (562, 265)
Y shape: (562, 52)
skipped: 0
saved: /content/q265_to_mp52_dataset.npz



##학습

In [ ]:
data = np.load("/content/q265_to_mp52_dataset.npz", allow_pickle=True)
X = data["X"]
Y = data["Y"]
blendshape_names = data["blendshape_names"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

reg = MLPRegressor(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    learning_rate_init=1e-3,
    batch_size=64,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=15,
    verbose=True,
    random_state=42,
)

reg.fit(X_train_s, y_train)

pred_val = np.clip(reg.predict(X_val_s), 0.0, 1.0)
mae = mean_absolute_error(y_val, pred_val)
print("val MAE:", float(mae))

joblib.dump(reg, "/content/q265_to_mp52_mlp.joblib")
joblib.dump(scaler, "/content/q265_scaler.joblib")

with open("/content/blendshape_names.json", "w") as f:
    json.dump(blendshape_names, f, ensure_ascii=False, indent=2)

print("saved: /content/q265_to_mp52_mlp.joblib")
print("saved: /content/q265_scaler.joblib")
print("saved: /content/blendshape_names.json")


Iteration 1, loss = 0.09610484
Validation score: -3459122176.000000
Iteration 2, loss = 0.02797409
Validation score: -1922958464.000000
Iteration 3, loss = 0.01579512
Validation score: -1341173248.000000
Iteration 4, loss = 0.01137493
Validation score: -1039023168.000000
Iteration 5, loss = 0.00915356
Validation score: -749971008.000000
Iteration 6, loss = 0.00755663
Validation score: -807979392.000000
Iteration 7, loss = 0.00660467
Validation score: -905042560.000000
Iteration 8, loss = 0.00587596
Validation score: -803173376.000000
Iteration 9, loss = 0.00529152
Validation score: -753972992.000000
Iteration 10, loss = 0.00484599
Validation score: -627476672.000000
Iteration 11, loss = 0.00448404
Validation score: -602637184.000000
Iteration 12, loss = 0.00418033
Validation score: -582732096.000000
Iteration 13, loss = 0.00392827
Validation score: -543021952.000000
Iteration 14, loss = 0.00371195
Validation score: -551191104.000000
Iteration 15, loss = 0.00353383
Validation score: -53

##새 이미지에서 추론

In [ ]:
from google.colab import files

reg = joblib.load("/content/q265_to_mp52_mlp.joblib")
scaler = joblib.load("/content/q265_scaler.joblib")

with open("/content/blendshape_names.json", "r") as f:
    blendshape_names = json.load(f)

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
image_rgb = read_rgb(img_path)

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    teacher = infer_mediapipe_teacher(landmarker, image_rgb)
    if teacher is None:
        raise RuntimeError("MediaPipe bbox extraction failed on this image.")

    pts, teacher_score_map, teacher_matrix = teacher
    bbox = get_face_bbox_from_landmarks(
        pts,
        image_rgb.shape,
        padding=0.25,
        make_square=True,
    )

x = infer_qualcomm_265(q_model, image_rgb, bbox)
pred = reg.predict(scaler.transform(x[None]))[0]
pred = np.clip(pred, 0.0, 1.0)

pred_map = {name: float(score) for name, score in zip(blendshape_names, pred)}
top_pred = sorted(pred_map.items(), key=lambda kv: kv[1], reverse=True)[:15]

print("Top predicted blendshapes")
for name, score in top_pred:
    print(f"{name:24s} {score:.4f}")

with open("/content/predicted_blendshapes.json", "w") as f:
    json.dump(pred_map, f, ensure_ascii=False, indent=2)

print("saved: /content/predicted_blendshapes.json")


Saving front.png to front.png
Top predicted blendshapes
eyeSquintRight           0.6865
eyeWideRight             0.4494
eyeLookUpLeft            0.3981
browOuterUpLeft          0.3162
browOuterUpRight         0.3136
eyeSquintLeft            0.3044
eyeLookOutRight          0.2968
mouthStretchLeft         0.2571
eyeLookUpRight           0.2424
noseSneerRight           0.2344
eyeLookInRight           0.2289
mouthPressLeft           0.2153
eyeWideLeft              0.2145
mouthSmileLeft           0.1983
eyeLookDownRight         0.1945
saved: /content/predicted_blendshapes.json


##비교

In [ ]:
teacher_vec = np.array([teacher_score_map[name] for name in blendshape_names], dtype=np.float32)
abs_err = np.abs(pred - teacher_vec)

pairs = list(zip(blendshape_names, pred, teacher_vec, abs_err))
pairs = sorted(pairs, key=lambda x: x[3], reverse=True)

print("Largest errors")
for name, p, t, e in pairs[:15]:
    print(f"{name:24s} pred={p:.4f} teacher={t:.4f} abs_err={e:.4f}")


Largest errors
eyeWideRight             pred=0.4494 teacher=0.0091 abs_err=0.4403
eyeSquintRight           pred=0.6865 teacher=0.2555 abs_err=0.4310
eyeLookUpLeft            pred=0.3981 teacher=0.1002 abs_err=0.2979
eyeLookOutRight          pred=0.2968 teacher=0.0213 abs_err=0.2755
mouthStretchLeft         pred=0.2571 teacher=0.0003 abs_err=0.2568
noseSneerRight           pred=0.2344 teacher=0.0000 abs_err=0.2344
browOuterUpRight         pred=0.3136 teacher=0.0808 abs_err=0.2328
browOuterUpLeft          pred=0.3162 teacher=0.0890 abs_err=0.2271
eyeWideLeft              pred=0.2145 teacher=0.0081 abs_err=0.2064
mouthPressLeft           pred=0.2153 teacher=0.0093 abs_err=0.2060
mouthSmileLeft           pred=0.1983 teacher=0.0002 abs_err=0.1981
cheekSquintLeft          pred=0.1769 teacher=0.0000 abs_err=0.1769
cheekPuff                pred=0.1576 teacher=0.0000 abs_err=0.1576
browDownRight            pred=0.1512 teacher=0.0097 abs_err=0.1415
mouthLeft                pred=0.1405 teacher=0.

#핵심

##import +모델/헬퍼함수

In [ ]:
TARGET_BLENDSHAPES = [
    "jawOpen",
    "mouthOpen",
    "mouthSmileLeft",
    "mouthSmileRight",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "browInnerUp",
    "browDownLeft",
    "browDownRight",
    "mouthPucker",
]


In [ ]:
import os
import glob
import json
import joblib
import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error

from qai_hub_models.models.facemap_3dmm.model import FaceMap_3DMM

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode


TARGET_BLENDSHAPES = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthPucker",
    "browInnerUp",
]


def create_face_landmarker(model_path="/content/face_landmarker.task", num_faces=1):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)


def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def collect_image_paths(root):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(paths)


def infer_mediapipe_teacher(landmarker, image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = landmarker.detect(mp_image)

    if not result.face_landmarks or not result.face_blendshapes:
        return None

    h, w = image_rgb.shape[:2]
    pts = np.array(
        [[lm.x * w, lm.y * h, lm.z * w] for lm in result.face_landmarks[0]],
        dtype=np.float32,
    )

    score_map = {c.category_name: float(c.score) for c in result.face_blendshapes[0]}

    matrix = None
    if result.facial_transformation_matrixes:
        matrix = np.array(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(-1)

    return pts, score_map, matrix


def get_face_bbox_from_landmarks(pts, image_shape, padding=0.25, make_square=True):
    h, w = image_shape[:2]

    xs = pts[:, 0]
    ys = pts[:, 1]

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0
    bh = y1 - y0

    x0 -= bw * padding
    x1 += bw * padding
    y0 -= bh * padding
    y1 += bh * padding

    if make_square:
        cx = (x0 + x1) / 2.0
        cy = (y0 + y1) / 2.0
        size = max(x1 - x0, y1 - y0)
        x0 = cx - size / 2.0
        x1 = cx + size / 2.0
        y0 = cy - size / 2.0
        y1 = cy + size / 2.0

    x0 = int(np.clip(np.floor(x0), 0, w - 1))
    y0 = int(np.clip(np.floor(y0), 0, h - 1))
    x1 = int(np.clip(np.ceil(x1), 0, w - 1))
    y1 = int(np.clip(np.ceil(y1), 0, h - 1))

    return x0, y0, x1, y1


def infer_qualcomm_265(model, image_rgb, bbox):
    x0, y0, x1, y1 = bbox
    crop = image_rgb[y0:y1 + 1, x0:x1 + 1]

    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy(crop).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        raw265 = model(inp)[0].cpu().numpy().astype(np.float32)

    return raw265


##데이터셋


In [ ]:
TRAIN_DIR = "/content/train_images"

image_paths = collect_image_paths(TRAIN_DIR)
print("num images:", len(image_paths))

q_model = FaceMap_3DMM.from_pretrained()

X_list = []
Y_list = []
used_paths = []
skipped = []

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(image_paths):
        try:
            image_rgb = read_rgb(path)

            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                skipped.append((path, "mediapipe_failed"))
                continue

            pts, score_map, matrix = teacher

            if not all(name in score_map for name in TARGET_BLENDSHAPES):
                skipped.append((path, "missing_blendshape_key"))
                continue

            y = np.array([score_map[name] for name in TARGET_BLENDSHAPES], dtype=np.float32)

            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                skipped.append((path, "qualcomm_failed"))
                continue

            X_list.append(x)
            Y_list.append(y)
            used_paths.append(path)

            if (i + 1) % 100 == 0:
                print(f"processed {i+1}/{len(image_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))

X = np.stack(X_list).astype(np.float32)
Y = np.stack(Y_list).astype(np.float32)

print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("targets:", TARGET_BLENDSHAPES)
print("skipped:", len(skipped))

np.savez(
    "/content/q265_to_mp7_dataset.npz",
    X=X,
    Y=Y,
    blendshape_names=np.array(TARGET_BLENDSHAPES, dtype=object),
    used_paths=np.array(used_paths, dtype=object),
)

with open("/content/skipped_subset.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("saved: /content/q265_to_mp7_dataset.npz")


num images: 562


processed 100/562
processed 200/562
processed 300/562
processed 400/562
processed 500/562
X shape: (562, 265)
Y shape: (562, 7)
targets: ['jawOpen', 'eyeBlinkLeft', 'eyeBlinkRight', 'mouthSmileLeft', 'mouthSmileRight', 'mouthPucker', 'browInnerUp']
skipped: 0
saved: /content/q265_to_mp7_dataset.npz



##학습

In [ ]:
data = np.load("/content/q265_to_mp7_dataset.npz", allow_pickle=True)
X = data["X"]
Y = data["Y"]
blendshape_names = data["blendshape_names"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

reg = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    learning_rate_init=1e-3,
    batch_size=32,
    max_iter=400,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    verbose=True,
    random_state=42,
)

reg.fit(X_train_s, y_train)

pred_val = np.clip(reg.predict(X_val_s), 0.0, 1.0)
mae = mean_absolute_error(y_val, pred_val)
print("val MAE:", float(mae))

for i, name in enumerate(blendshape_names):
    mae_i = np.mean(np.abs(pred_val[:, i] - y_val[:, i]))
    print(f"{name:20s} mae={mae_i:.4f}")

joblib.dump(reg, "/content/q265_to_mp10_mlp.joblib")
joblib.dump(scaler, "/content/q265_scaler_10.joblib")

with open("/content/blendshape_names_10.json", "w") as f:
    json.dump(blendshape_names, f, ensure_ascii=False, indent=2)

print("saved: /content/q265_to_mp10_mlp.joblib")


Iteration 1, loss = 0.11145211
Validation score: -2.320041
Iteration 2, loss = 0.02948738
Validation score: -0.513699
Iteration 3, loss = 0.01526092
Validation score: 0.053997
Iteration 4, loss = 0.01019657
Validation score: 0.269785
Iteration 5, loss = 0.00700342
Validation score: 0.445306
Iteration 6, loss = 0.00542474
Validation score: 0.533387
Iteration 7, loss = 0.00455889
Validation score: 0.577639
Iteration 8, loss = 0.00394769
Validation score: 0.609254
Iteration 9, loss = 0.00365417
Validation score: 0.577177
Iteration 10, loss = 0.00376385
Validation score: 0.568259
Iteration 11, loss = 0.00346393
Validation score: 0.637080
Iteration 12, loss = 0.00279556
Validation score: 0.635028
Iteration 13, loss = 0.00271860
Validation score: 0.649034
Iteration 14, loss = 0.00239016
Validation score: 0.626015
Iteration 15, loss = 0.00227699
Validation score: 0.657706
Iteration 16, loss = 0.00218148
Validation score: 0.646634
Iteration 17, loss = 0.00238680
Validation score: 0.696400
Iter

In [ ]:
NEUTRAL_DIR = "/content/neutral_images"

neutral_paths = collect_image_paths(NEUTRAL_DIR)
print("num neutral images:", len(neutral_paths))

reg = joblib.load("/content/q265_to_mp10_mlp.joblib")
scaler = joblib.load("/content/q265_scaler_10.joblib")

neutral_preds = []

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for path in neutral_paths:
        try:
            image_rgb = read_rgb(path)
            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                continue

            pts, score_map, matrix = teacher
            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                continue

            pred = reg.predict(scaler.transform(x[None]))[0]
            neutral_preds.append(pred)
        except:
            pass

neutral_bias = np.mean(np.stack(neutral_preds), axis=0) if neutral_preds else np.zeros(len(TARGET_BLENDSHAPES), dtype=np.float32)
np.save("/content/neutral_bias_7.npy", neutral_bias)

print("neutral_bias:")
for name, val in zip(TARGET_BLENDSHAPES, neutral_bias):
    print(f"{name:18s} {float(val):.4f}")


num neutral images: 0
neutral_bias:
jawOpen            0.0000
eyeBlinkLeft       0.0000
eyeBlinkRight      0.0000
mouthSmileLeft     0.0000
mouthSmileRight    0.0000
mouthPucker        0.0000
browInnerUp        0.0000


##새 이미지에서 추론

In [ ]:
from google.colab import files

reg = joblib.load("/content/q265_to_mp10_mlp.joblib")
scaler = joblib.load("/content/q265_scaler_10.joblib")

with open("/content/blendshape_names_10.json", "r") as f:
    blendshape_names = json.load(f)

if os.path.exists("/content/neutral_bias_7.npy"):
    neutral_bias = np.load("/content/neutral_bias_7.npy")
else:
    neutral_bias = np.zeros(len(blendshape_names), dtype=np.float32)

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
image_rgb = read_rgb(img_path)

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    teacher = infer_mediapipe_teacher(landmarker, image_rgb)
    if teacher is None:
        raise RuntimeError("MediaPipe failed on this image.")

    pts, teacher_score_map, teacher_matrix = teacher
    bbox = get_face_bbox_from_landmarks(
        pts,
        image_rgb.shape,
        padding=0.25,
        make_square=True,
    )

x = infer_qualcomm_265(q_model, image_rgb, bbox)
pred = reg.predict(scaler.transform(x[None]))[0]

pred = np.clip(pred - neutral_bias, 0.0, 1.0)
pred[pred < 0.05] = 0.0

teacher_vec = np.array([teacher_score_map[name] for name in blendshape_names], dtype=np.float32)

for i, name in enumerate(blendshape_names):
    print(
        f"{name:18s} pred={pred[i]:.4f} "
        f"teacher={teacher_vec[i]:.4f} "
        f"abs_err={abs(pred[i] - teacher_vec[i]):.4f}"
    )


Saving front.png to front.png
jawOpen            pred=0.0000 teacher=0.0022 abs_err=0.0022
eyeBlinkLeft       pred=0.3323 teacher=0.0892 abs_err=0.2431
eyeBlinkRight      pred=0.0000 teacher=0.0482 abs_err=0.0482
mouthSmileLeft     pred=0.0908 teacher=0.0002 abs_err=0.0906
mouthSmileRight    pred=0.5635 teacher=0.0003 abs_err=0.5633
mouthPucker        pred=0.0000 teacher=0.1971 abs_err=0.1971
browInnerUp        pred=0.0689 teacher=0.2035 abs_err=0.1346


In [ ]:
raw_pred = reg.predict(scaler.transform(x[None]))[0]
bias_pred = np.clip(raw_pred - neutral_bias, 0.0, 1.0)
final_pred = bias_pred.copy()
final_pred[final_pred < 0.05] = 0.0

for i, name in enumerate(blendshape_names):
    print(
        f"{name:18s} raw={raw_pred[i]:.4f} "
        f"bias={bias_pred[i]:.4f} "
        f"final={final_pred[i]:.4f} "
        f"teacher={teacher_vec[i]:.4f}"
    )


jawOpen            raw=-0.5370 bias=0.0000 final=0.0000 teacher=0.0022
eyeBlinkLeft       raw=0.3323 bias=0.3323 final=0.3323 teacher=0.0892
eyeBlinkRight      raw=-0.1507 bias=0.0000 final=0.0000 teacher=0.0482
mouthSmileLeft     raw=0.0908 bias=0.0908 final=0.0908 teacher=0.0002
mouthSmileRight    raw=0.5635 bias=0.5635 final=0.5635 teacher=0.0003
mouthPucker        raw=0.0096 bias=0.0096 final=0.0000 teacher=0.1971
browInnerUp        raw=0.0689 bias=0.0689 final=0.0689 teacher=0.2035


##저장

In [ ]:
pred_map = {name: float(score) for name, score in zip(blendshape_names, pred)}

with open("/content/predicted_blendshapes_10.json", "w") as f:
    json.dump(pred_map, f, ensure_ascii=False, indent=2)

print("saved: /content/predicted_blendshapes_10.json")
pred_map


saved: /content/predicted_blendshapes_10.json


{'jawOpen': 0.0,
 'eyeBlinkLeft': 0.33227047324180603,
 'eyeBlinkRight': 0.0,
 'mouthSmileLeft': 0.09080862998962402,
 'mouthSmileRight': 0.563539981842041,
 'mouthPucker': 0.0,
 'browInnerUp': 0.06886918842792511}

In [ ]:
reg = joblib.load("/content/q265_to_mp10_mlp.joblib")
scaler = joblib.load("/content/q265_scaler_10.joblib")

with open("/content/blendshape_names_10.json", "r") as f:
    blendshape_names = json.load(f)

if os.path.exists("/content/neutral_bias_10.npy"):
    neutral_bias = np.load("/content/neutral_bias_10.npy")
else:
    neutral_bias = np.zeros(len(blendshape_names), dtype=np.float32)

image_paths = collect_image_paths("/content/train_images")
results = []

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(image_paths):
        try:
            image_rgb = read_rgb(path)
            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                continue

            pts, score_map, matrix = teacher
            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                continue

            pred = reg.predict(scaler.transform(x[None]))[0]
            pred = np.clip(pred - neutral_bias, 0.0, 1.0)
            pred[pred < 0.05] = 0.0

            pred_map = {name: float(score) for name, score in zip(blendshape_names, pred)}
            results.append({
                "path": path,
                "blendshapes": pred_map,
            })

            if (i + 1) % 100 == 0:
                print(f"done {i+1}/{len(image_paths)}")
        except Exception:
            pass

with open("/content/predicted_blendshapes_seq.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("saved: /content/predicted_blendshapes_seq.json")


done 100/562
done 200/562
done 300/562
done 400/562
done 500/562
saved: /content/predicted_blendshapes_seq.json


#test


In [ ]:
import os
import json
import joblib
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

data = np.load("/content/q265_to_mp7_dataset.npz", allow_pickle=True)
X = data["X"]
Y = data["Y"]
blendshape_names = data["blendshape_names"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

models = {}
pred_val = np.zeros_like(y_val)

for i, name in enumerate(blendshape_names):
    model = Ridge(alpha=3.0)
    model.fit(X_train_s, y_train[:, i])

    pred_i = model.predict(X_val_s)
    pred_i = np.clip(pred_i, 0.0, 1.0)

    pred_val[:, i] = pred_i
    models[name] = model

    mae_i = mean_absolute_error(y_val[:, i], pred_i)
    print(f"{name:18s} mae={mae_i:.4f}")

overall_mae = mean_absolute_error(y_val, pred_val)
print("overall MAE:", overall_mae)

os.makedirs("/content/q265_mp7_ridge_models", exist_ok=True)

for name, model in models.items():
    joblib.dump(model, f"/content/q265_mp7_ridge_models/{name}.joblib")

joblib.dump(scaler, "/content/q265_mp7_ridge_models/scaler.joblib")

with open("/content/q265_mp7_ridge_models/blendshape_names.json", "w") as f:
    json.dump(blendshape_names, f, ensure_ascii=False, indent=2)

print("saved models -> /content/q265_mp7_ridge_models")


jawOpen            mae=0.0165
eyeBlinkLeft       mae=0.0249
eyeBlinkRight      mae=0.0278
mouthSmileLeft     mae=0.0491
mouthSmileRight    mae=0.0526
mouthPucker        mae=0.0328
browInnerUp        mae=0.0410
overall MAE: 0.03497243672609329
saved models -> /content/q265_mp7_ridge_models


In [ ]:
import os
import json
import joblib
import numpy as np
from google.colab import files

with open("/content/q265_mp7_ridge_models/blendshape_names.json", "r") as f:
    blendshape_names = json.load(f)

scaler = joblib.load("/content/q265_mp7_ridge_models/scaler.joblib")
models = {
    name: joblib.load(f"/content/q265_mp7_ridge_models/{name}.joblib")
    for name in blendshape_names
}

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
image_rgb = read_rgb(img_path)

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    teacher = infer_mediapipe_teacher(landmarker, image_rgb)
    if teacher is None:
        raise RuntimeError("MediaPipe failed on this image.")

    pts, teacher_score_map, teacher_matrix = teacher
    bbox = get_face_bbox_from_landmarks(
        pts,
        image_rgb.shape,
        padding=0.25,
        make_square=True,
    )

x = infer_qualcomm_265(q_model, image_rgb, bbox)
x_s = scaler.transform(x[None])

pred = []
for name in blendshape_names:
    val = models[name].predict(x_s)[0]
    val = float(np.clip(val, 0.0, 1.0))
    if val < 0.02:
        val = 0.0
    pred.append(val)

pred = np.array(pred, dtype=np.float32)
teacher_vec = np.array([teacher_score_map[name] for name in blendshape_names], dtype=np.float32)

for i, name in enumerate(blendshape_names):
    print(
        f"{name:18s} pred={pred[i]:.4f} "
        f"teacher={teacher_vec[i]:.4f} "
        f"abs_err={abs(pred[i] - teacher_vec[i]):.4f}"
    )


Saving front.png to front (4).png
jawOpen            pred=0.0000 teacher=0.0022 abs_err=0.0022
eyeBlinkLeft       pred=0.0441 teacher=0.0891 abs_err=0.0450
eyeBlinkRight      pred=0.0375 teacher=0.0481 abs_err=0.0107
mouthSmileLeft     pred=0.0000 teacher=0.0002 abs_err=0.0002
mouthSmileRight    pred=0.0000 teacher=0.0003 abs_err=0.0003
mouthPucker        pred=0.1762 teacher=0.1970 abs_err=0.0208
browInnerUp        pred=0.0000 teacher=0.2035 abs_err=0.2035


In [ ]:
import json
import numpy as np
import joblib
from google.colab import files

TARGET_BLENDSHAPES = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthPucker",
]

# 모델 로드
scaler = joblib.load("/content/q265_mp7_ridge_models/scaler.joblib")
models = {
    name: joblib.load(f"/content/q265_mp7_ridge_models/{name}.joblib")
    for name in TARGET_BLENDSHAPES
}

uploaded = files.upload()
img_path = next(iter(uploaded.keys()))
image_rgb = read_rgb(img_path)

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    teacher = infer_mediapipe_teacher(landmarker, image_rgb)
    if teacher is None:
        raise RuntimeError("MediaPipe failed on this image.")

    pts, teacher_score_map, teacher_matrix = teacher
    bbox = get_face_bbox_from_landmarks(
        pts,
        image_rgb.shape,
        padding=0.25,
        make_square=True,
    )

x = infer_qualcomm_265(q_model, image_rgb, bbox)
x_s = scaler.transform(x[None])

pred_map = {}
for name in TARGET_BLENDSHAPES:
    val = float(np.clip(models[name].predict(x_s)[0], 0.0, 1.0))
    if val < 0.02:
        val = 0.0
    pred_map[name] = val

# MediaPipe demo 스타일 비슷하게 category_name/score 배열로 저장
blendshape_json = {
    "faceBlendshapes": [
        {
            "categoryName": name,
            "score": pred_map[name],
        }
        for name in TARGET_BLENDSHAPES
    ]
}

with open("/content/avatar_blendshapes.json", "w") as f:
    json.dump(blendshape_json, f, ensure_ascii=False, indent=2)

print(json.dumps(blendshape_json, ensure_ascii=False, indent=2))
print("saved: /content/avatar_blendshapes.json")


Saving close.jpg to close.jpg
{
  "faceBlendshapes": [
    {
      "categoryName": "jawOpen",
      "score": 0.12206122279167175
    },
    {
      "categoryName": "eyeBlinkLeft",
      "score": 0.7554131746292114
    },
    {
      "categoryName": "eyeBlinkRight",
      "score": 0.7611514329910278
    },
    {
      "categoryName": "mouthSmileLeft",
      "score": 0.0
    },
    {
      "categoryName": "mouthSmileRight",
      "score": 0.04969361424446106
    },
    {
      "categoryName": "mouthPucker",
      "score": 0.08726927638053894
    }
  ]
}
saved: /content/avatar_blendshapes.json


In [ ]:
simple_json = {
    "blendshapes": pred_map,
    "poseMatrix": teacher_matrix.tolist() if teacher_matrix is not None else None,
}

with open("/content/avatar_simple.json", "w") as f:
    json.dump(simple_json, f, ensure_ascii=False, indent=2)

print(json.dumps(simple_json, ensure_ascii=False, indent=2))
print("saved: /content/avatar_simple.json")


{
  "blendshapes": {
    "jawOpen": 0.12206122279167175,
    "eyeBlinkLeft": 0.7554131746292114,
    "eyeBlinkRight": 0.7611514329910278,
    "mouthSmileLeft": 0.0,
    "mouthSmileRight": 0.04969361424446106,
    "mouthPucker": 0.08726927638053894
  },
  "poseMatrix": [
    0.9853284955024719,
    -0.16395699977874756,
    0.04738291725516319,
    0.7399170398712158,
    0.17065328359603882,
    0.949964702129364,
    -0.2616179585456848,
    3.3975465297698975,
    -0.002118009841069579,
    0.26586559414863586,
    0.964007556438446,
    -33.5713005065918,
    0.0,
    0.0,
    0.0,
    1.0
  ]
}
saved: /content/avatar_simple.json


In [ ]:
!mkdir -p /content/video_frames
!ffmpeg -i /content/d0.mp4 -vf fps=3 -q:v 2 /content/video_frames/frame_%05d.jpg
!ls /content/video_frames | head
!ls /content/video_frames | wc -l


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
import os
import json
import joblib
import numpy as np

TARGET_BLENDSHAPES = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthPucker",
]

scaler = joblib.load("/content/q265_mp7_ridge_models/scaler.joblib")
models = {
    name: joblib.load(f"/content/q265_mp7_ridge_models/{name}.joblib")
    for name in TARGET_BLENDSHAPES
}

frame_paths = collect_image_paths("/content/video_frames")
print("num frames:", len(frame_paths))

results = []
skipped = []

fps_value = 3.0

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(frame_paths):
        try:
            image_rgb = read_rgb(path)

            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                skipped.append((path, "mediapipe_failed"))
                continue

            pts, teacher_score_map, teacher_matrix = teacher

            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                skipped.append((path, "qualcomm_failed"))
                continue

            x_s = scaler.transform(x[None])

            pred_map = {}
            for name in TARGET_BLENDSHAPES:
                val = float(np.clip(models[name].predict(x_s)[0], 0.0, 1.0))
                if val < 0.02:
                    val = 0.0
                pred_map[name] = val

            results.append({
                "frame_index": i,
                "time_sec": i / fps_value,
                "blendshapes": pred_map,
                "poseMatrix": teacher_matrix.tolist() if teacher_matrix is not None else None,
                "bbox": [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])],
            })

            if (i + 1) % 100 == 0:
                print(f"done {i+1}/{len(frame_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))

payload = {
    "fps": fps_value,
    "frame_size": {
        "width": int(read_rgb(frame_paths[0]).shape[1]),
        "height": int(read_rgb(frame_paths[0]).shape[0]),
    },
    "frames": results,
}

with open("/content/raccoon_video_blendshapes.json", "w") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

with open("/content/raccoon_video_blendshapes_skipped.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("saved: /content/raccoon_video_blendshapes.json")
print("saved: /content/raccoon_video_blendshapes_skipped.json")
print("usable frames:", len(results))
print("skipped frames:", len(skipped))


num frames: 9
saved: /content/raccoon_video_blendshapes.json
saved: /content/raccoon_video_blendshapes_skipped.json
usable frames: 9
skipped frames: 0


In [ ]:
from google.colab import files
files.download("/content/raccoon_video_blendshapes.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

#영상 메타

In [ ]:
import json
import numpy as np
import joblib
from pathlib import Path

# 네가 실제로 학습한 타깃 이름으로 맞추기
TARGET_BLENDSHAPES = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthPucker",
]

METADATA_PATH = "/content/metadata_1774237242321.json"
OUTPUT_PATH = "/content/raccoon_from_metadata.json"

# 네 코랩에서 저장한 모델 경로
SCALER_PATH = "/content/q265_mp7_ridge_models/scaler.joblib"
MODEL_DIR = "/content/q265_mp7_ridge_models"

scaler = joblib.load(SCALER_PATH)
models = {
    name: joblib.load(f"{MODEL_DIR}/{name}.joblib")
    for name in TARGET_BLENDSHAPES
}

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

frames = metadata["frames"]

def pick_main_face(faces):
    if not faces:
        return None
    # 일단 가장 큰 얼굴 사용
    return max(
        faces,
        key=lambda face: face.get("bbox", {}).get("width", 0) * face.get("bbox", {}).get("height", 0)
    )

def bbox_to_xyxy(bbox):
    if not bbox:
        return None
    x = float(bbox["x"])
    y = float(bbox["y"])
    w = float(bbox["width"])
    h = float(bbox["height"])
    return [x, y, x + w, y + h]

# fps 추정
pts_list = [fr["pts_us"] for fr in frames if "pts_us" in fr]
if len(pts_list) >= 2:
    deltas = np.diff(np.array(pts_list, dtype=np.int64))
    deltas = deltas[deltas > 0]
    fps = float(1_000_000.0 / np.median(deltas)) if len(deltas) else 30.0
else:
    fps = 30.0

start_pts = frames[0]["pts_us"] if frames else 0

results = []

for i, frame in enumerate(frames):
    faces = frame.get("faces", [])
    face = pick_main_face(faces)

    if face is None:
        results.append({
            "frame_index": i,
            "time_sec": float((frame["pts_us"] - start_pts) / 1_000_000.0),
            "has_face": False,
            "blendshapes": {name: 0.0 for name in TARGET_BLENDSHAPES},
        })
        continue

    coeff = np.asarray(face["tdmm_raw"]["coeffs"], dtype=np.float32).reshape(-1)

    # 네 student 모델이 265 입력으로 학습됐으면 그대로 사용
    if coeff.shape[0] < 265:
        results.append({
            "frame_index": i,
            "time_sec": float((frame["pts_us"] - start_pts) / 1_000_000.0),
            "has_face": False,
            "blendshapes": {name: 0.0 for name in TARGET_BLENDSHAPES},
        })
        continue

    x = coeff[:265]
    x_s = scaler.transform(x[None])

    pred_map = {}
    for name in TARGET_BLENDSHAPES:
        val = float(np.clip(models[name].predict(x_s)[0], 0.0, 1.0))
        if val < 0.02:
            val = 0.0
        pred_map[name] = val

    pose = {
        "pitch": float(coeff[258]) * np.pi / 2.0,
        "yaw": float(coeff[259]) * np.pi / 2.0,
        "roll": float(coeff[260]) * np.pi / 2.0,
    }

    results.append({
        "frame_index": i,
        "time_sec": float((frame["pts_us"] - start_pts) / 1_000_000.0),
        "pts_us": int(frame["pts_us"]),
        "has_face": True,
        "tracking_id": int(face.get("tracking_id", -1)),
        "blendshapes": pred_map,
        "pose_radians": pose,
        "det_bbox_xyxy": bbox_to_xyxy(face.get("bbox")),
    })

payload = {
    "fps": fps,
    "frames": results,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("saved:", OUTPUT_PATH)
print("fps:", fps)
print("num frames:", len(results))
print("frames with face:", sum(1 for r in results if r["has_face"]))


FileNotFoundError: [Errno 2] No such file or directory: '/content/q265_mp7_ridge_models/scaler.joblib'

#영상에서


In [ ]:
!mkdir -p /content/video_frames
!ffmpeg -i /content/input.mp4 -vf fps=3 -q:v 2 /content/video_frames/frame_%05d.jpg
!ls /content/video_frames | head
!ls /content/video_frames | wc -l

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
import os
import glob
import json
import joblib
import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from qai_hub_models.models.facemap_3dmm.model import FaceMap_3DMM

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

TARGET_BLENDSHAPES = [
    "jawOpen",
    "eyeBlinkLeft",
    "eyeBlinkRight",
    "mouthSmileLeft",
    "mouthSmileRight",
    "mouthPucker",
]

def create_face_landmarker(model_path="/content/face_landmarker.task", num_faces=1):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)

def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def collect_image_paths(root):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(paths)

def infer_mediapipe_teacher(landmarker, image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = landmarker.detect(mp_image)

    if not result.face_landmarks or not result.face_blendshapes:
        return None

    h, w = image_rgb.shape[:2]
    pts = np.array(
        [[lm.x * w, lm.y * h, lm.z * w] for lm in result.face_landmarks[0]],
        dtype=np.float32,
    )

    score_map = {c.category_name: float(c.score) for c in result.face_blendshapes[0]}

    matrix = None
    if result.facial_transformation_matrixes:
        matrix = np.array(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(-1)

    return pts, score_map, matrix

def get_face_bbox_from_landmarks(pts, image_shape, padding=0.25, make_square=True):
    h, w = image_shape[:2]

    xs = pts[:, 0]
    ys = pts[:, 1]

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0
    bh = y1 - y0

    x0 -= bw * padding
    x1 += bw * padding
    y0 -= bh * padding
    y1 += bh * padding

    if make_square:
        cx = (x0 + x1) / 2.0
        cy = (y0 + y1) / 2.0
        size = max(x1 - x0, y1 - y0)
        x0 = cx - size / 2.0
        x1 = cx + size / 2.0
        y0 = cy - size / 2.0
        y1 = cy + size / 2.0

    x0 = int(np.clip(np.floor(x0), 0, w - 1))
    y0 = int(np.clip(np.floor(y0), 0, h - 1))
    x1 = int(np.clip(np.ceil(x1), 0, w - 1))
    y1 = int(np.clip(np.ceil(y1), 0, h - 1))

    return x0, y0, x1, y1

def infer_qualcomm_265(model, image_rgb, bbox):
    x0, y0, x1, y1 = bbox
    crop = image_rgb[y0:y1 + 1, x0:x1 + 1]

    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy(crop).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        raw265 = model(inp)[0].cpu().numpy().astype(np.float32)

    return raw265

q_model = FaceMap_3DMM.from_pretrained()
scaler = joblib.load("/content/q265_scaler_10.joblib")
models = {
    name: joblib.load(f"/content/q265_to_mp10_mlp.joblib")
    for name in TARGET_BLENDSHAPES
}

print("ready")


ready


In [ ]:
frame_paths = collect_image_paths("/content/video_frames")
print("num frames:", len(frame_paths))

results = []
skipped = []

fps_value = 3.0
THRESHOLD = 0.02

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(frame_paths):
        try:
            image_rgb = read_rgb(path)

            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                results.append({
                    "frame_index": i,
                    "time_sec": i / fps_value,
                    "has_face": False,
                    "blendshapes": {name: 0.0 for name in TARGET_BLENDSHAPES},
                })
                continue

            pts, teacher_score_map, teacher_matrix = teacher

            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                results.append({
                    "frame_index": i,
                    "time_sec": i / fps_value,
                    "has_face": False,
                    "blendshapes": {name: 0.0 for name in TARGET_BLENDSHAPES},
                })
                continue

            x_s = scaler.transform(x[None])

            pred_map = {}
            for name in TARGET_BLENDSHAPES:
                val = float(np.clip(models[name].predict(x_s)[0], 0.0, 1.0))
                if val < THRESHOLD:
                    val = 0.0
                pred_map[name] = val

            pose = {
                "pitch": float(x[258]) * float(np.pi / 2.0),
                "yaw": float(x[259]) * float(np.pi / 2.0),
                "roll": float(x[260]) * float(np.pi / 2.0),
            }

            results.append({
                "frame_index": i,
                "time_sec": i / fps_value,
                "has_face": True,
                "blendshapes": pred_map,
                "pose_radians": pose,
                "det_bbox_xyxy": [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])],
            })

            if (i + 1) % 100 == 0:
                print(f"done {i+1}/{len(frame_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))
            results.append({
                "frame_index": i,
                "time_sec": i / fps_value,
                "has_face": False,
                "blendshapes": {name: 0.0 for name in TARGET_BLENDSHAPES},
            })

payload = {
    "fps": fps_value,
    "frame_size": {
        "width": int(read_rgb(frame_paths[0]).shape[1]),
        "height": int(read_rgb(frame_paths[0]).shape[0]),
    },
    "frames": results,
}

with open("/content/raccoon_video_blendshapes.json", "w") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

with open("/content/raccoon_video_blendshapes_skipped.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("saved: /content/raccoon_video_blendshapes.json")
print("saved: /content/raccoon_video_blendshapes_skipped.json")
print("usable frames:", sum(1 for r in results if r["has_face"]))
print("total frames:", len(results))
print("skipped errors:", len(skipped))


num frames: 562
saved: /content/raccoon_video_blendshapes.json
saved: /content/raccoon_video_blendshapes_skipped.json
usable frames: 0
total frames: 562
skipped errors: 562


In [ ]:
from google.colab import files
files.download("/content/raccoon_video_blendshapes.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>